# Standard LIBERO after the 10-action fix

Compact, zero-GPU analysis of the corrected standard-LIBERO experiment. This notebook is locked to `libero-hybrid-schedules-k3-a10-v2`, rejects rows whose logged execution horizon is not exactly 10 actions, and requires the complete 1,920-rollout / 400-identity matrix.

It reports baseline success and failure-detection AUC, exact-paired SR changes, first-k and individual-chunk AUC, and exploratory uncertainty-window sweeps. Every window-policy SR and delta uses the **entire matched cohort** as its denominator. Prefix/window results are post-hoc diagnostics, not prospective online policies.

## 1. Setup

In [ ]:
EXTRAS = 'analysis'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Fetch and strictly validate the corrected cohort

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from analysis.conditions import PAIR_KEYS, condition_label
from analysis.standard_libero import detector_tables, paired_comparisons, success_tables
from analysis.statistics import bootstrap_auc, paired_bootstrap_ci, paired_counts
from analysis.validate import pair_one_to_one, validate_standard
from pnp.config import Method
from pnp.experiments import LIBERO_10STEP_EXPERIMENT
from pnp.store import SupabaseStore

EXPERIMENT = LIBERO_10STEP_EXPERIMENT
EXPECTED_N_ACTION_STEPS = 10
OUTPUT = Path('standard_libero_10_action_outputs')
OUTPUT.mkdir(exist_ok=True)
store = SupabaseStore()

rollouts = pd.DataFrame(store.fetch_all(
    'rollouts', '*',
    configure=lambda q: q.eq('experiment', EXPERIMENT),
    order_by=('rollout_id',)))
runs = pd.DataFrame(store.fetch_all(
    'experiment_runs', '*',
    configure=lambda q: q.eq('experiment', EXPERIMENT),
    order_by=('run_id',)))
assert len(rollouts), f'No rows found for {EXPERIMENT}'
assert set(rollouts.experiment.astype(str)) == {EXPERIMENT}

def as_config(value):
    if isinstance(value, str):
        value = json.loads(value)
    return value or {}

rollouts['n_action_steps_logged'] = rollouts.config_json.apply(
    lambda value: as_config(value).get('n_action_steps'))
logged_horizons = set(rollouts.n_action_steps_logged.dropna().astype(int))
assert logged_horizons == {EXPECTED_N_ACTION_STEPS}, (
    f'Expected only 10-action rows; found horizons {sorted(logged_horizons)}')
assert rollouts.n_action_steps_logged.notna().all(), 'Some rows do not log n_action_steps'

validated, validation = validate_standard(rollouts, runs)
print({'experiment': EXPERIMENT,
       'n_action_steps': EXPECTED_N_ACTION_STEPS,
       'rollouts': validation['n_rollouts'],
       'identities': validation['n_identities'],
       'configurations': validation['n_configurations']})
for warning in validation['warnings']:
    print('WARNING:', warning)
coverage = pd.DataFrame(validation['coverage'])
display(coverage[['condition_label', 'n_rollouts', 'full_ablation_n', 'broad_validation_n']])

observed = validated[validated.method.eq(Method.UNCERTAINTY)].copy()
observed_ids = observed.rollout_id.astype(str).tolist()
step_rows = []
for start in range(0, len(observed_ids), 80):
    batch = observed_ids[start:start + 80]
    step_rows.extend(store.fetch_all(
        'pnp_euler_steps', 'rollout_id,chunk_idx,euler_step,u_mean',
        configure=lambda q, ids=batch: q.in_('rollout_id', ids),
        order_by=('rollout_id', 'chunk_idx', 'euler_step')))
steps = pd.DataFrame(step_rows)
assert len(steps), 'No pnp_euler_steps found for the observed baseline'
assert steps.rollout_id.nunique() == 400, (
    f'Expected telemetry for 400 baseline episodes, found {steps.rollout_id.nunique()}')
print(f'Loaded {len(steps):,} uncertainty-step rows for 400 baseline episodes.')

## 3. Baseline SR, failure AUC, and paired SR deltas

In [ ]:
tables = {**success_tables(validated),
          'paired_comparisons': paired_comparisons(validated),
          **detector_tables(validated, steps)}

print('Observed/no-op baseline: uncertainty as a failure detector')
display(tables['detector_summary'][[
    'estimate_scope', 'n', 'failures', 'roc_auc', 'pr_auc',
    'roc_ci_low', 'roc_ci_high']].round(4))
auc_by_suite = tables['detector_by_suite'].copy()
display(auc_by_suite[[
    'suite', 'n', 'failures', 'roc_auc', 'roc_ci_low', 'roc_ci_high', 'pr_auc']].round(4))

baseline_sr_by_suite = tables['observed_success_by_suite'].copy()
print('Observed/no-op baseline SR by suite')
display(baseline_sr_by_suite[['suite', 'n', 'successes', 'sr', 'ci_low', 'ci_high']].round(4))

overall_deltas = tables['paired_comparisons'].copy()
print('Exact-paired SR deltas; all_identity covers all 400 episodes, full_ablation covers the fixed 80')
display(overall_deltas[[
    'cohort', 'condition_label', 'n', 'baseline_sr', 'condition_sr', 'delta_pp',
    'delta_ci_low_pp', 'delta_ci_high_pp', 'F_to_S', 'S_to_F', 'p_raw']].round(4))

per_suite_rows = []
full_hashes = [h for h, g in validated.groupby('config_hash')
               if len(g) == 400 and not g.method.eq(Method.UNCERTAINTY).all()]
for config_hash in full_hashes:
    condition = validated[validated.config_hash.eq(config_hash)]
    label = condition_label(condition.iloc[0])
    paired = pair_one_to_one(observed, condition)
    for suite, group in paired.groupby('suite', sort=True):
        b = group.success_baseline.astype(bool).to_numpy()
        c = group.success_condition.astype(bool).to_numpy()
        lo, hi = paired_bootstrap_ci(b, c, n_boot=5000)
        per_suite_rows.append({
            'suite': suite, 'condition_label': label, 'n': len(group),
            'baseline_sr': b.mean(), 'condition_sr': c.mean(),
            'delta_pp': 100 * (c.mean() - b.mean()),
            'delta_ci_low_pp': 100 * lo, 'delta_ci_high_pp': 100 * hi,
            **paired_counts(b, c)})
per_suite_deltas = pd.DataFrame(per_suite_rows)
print('Per-suite deltas for the two conditions collected on all 400 episodes')
display(per_suite_deltas.round(4))

for name, table in {
    'baseline_auc_by_suite': auc_by_suite,
    'baseline_sr_by_suite': baseline_sr_by_suite,
    'overall_paired_deltas': overall_deltas,
    'per_suite_paired_deltas': per_suite_deltas,
}.items():
    table.to_csv(OUTPUT / f'{name}.csv', index=False)

In [ ]:
suite_order = baseline_sr_by_suite.sort_values('suite').suite.tolist()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sr_plot = baseline_sr_by_suite.set_index('suite').loc[suite_order]
axes[0].bar(range(len(sr_plot)), 100 * sr_plot.sr, color='#4C78A8')
axes[0].set(xticks=range(len(sr_plot)), xticklabels=sr_plot.index, ylim=(0, 100),
            ylabel='Success rate (%)', title='Corrected 10-action baseline SR')
axes[0].tick_params(axis='x', rotation=25)
auc_plot = auc_by_suite.set_index('suite').loc[suite_order]
valid_auc = auc_plot.roc_auc.notna()
x = np.arange(len(auc_plot))
axes[1].bar(x[valid_auc], auc_plot.loc[valid_auc, 'roc_auc'], color='#F58518')
axes[1].errorbar(
    x[valid_auc], auc_plot.loc[valid_auc, 'roc_auc'],
    yerr=np.vstack([auc_plot.loc[valid_auc, 'roc_auc'] - auc_plot.loc[valid_auc, 'roc_ci_low'],
                    auc_plot.loc[valid_auc, 'roc_ci_high'] - auc_plot.loc[valid_auc, 'roc_auc']]),
    fmt='none', ecolor='black', capsize=3)
axes[1].axhline(.5, color='black', linestyle='--', linewidth=1, label='chance')
axes[1].set(xticks=x, xticklabels=auc_plot.index, ylim=(0, 1),
            ylabel='ROC-AUC (uncertainty predicts failure)', title='Failure AUC by suite')
axes[1].tick_params(axis='x', rotation=25)
axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT / 'baseline_sr_and_auc_by_suite.png', dpi=180, bbox_inches='tight')
plt.show()

roc = tables['detector_roc_curves']
fig, ax = plt.subplots(figsize=(6, 5))
for suite, group in roc.groupby('suite', sort=True):
    ax.plot(group.fpr, group.tpr, linewidth=2 if suite == 'pooled' else 1.3, label=suite)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
ax.set(xlabel='False-positive rate', ylabel='True-positive rate',
       title='Uncertainty as a failure detector', xlim=(0, 1), ylim=(0, 1))
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT / 'baseline_failure_roc.png', dpi=180, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(11, 4.5))
labels = per_suite_deltas.condition_label.unique().tolist()
width = .8 / len(labels)
x = np.arange(len(suite_order))
for i, label in enumerate(labels):
    group = per_suite_deltas[per_suite_deltas.condition_label.eq(label)].set_index('suite').loc[suite_order]
    ax.bar(x - .4 + width / 2 + i * width, group.delta_pp, width, label=label)
ax.axhline(0, color='black', linewidth=1)
ax.set(xticks=x, xticklabels=suite_order, ylabel='Paired SR change (percentage points)',
       title='Corrected 10-action SR change by suite')
ax.tick_params(axis='x', rotation=25)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT / 'paired_sr_delta_by_suite.png', dpi=180, bbox_inches='tight')
plt.show()

## 4. Chunk tests

`prefix_k_chunks` averages baseline uncertainty over the first `min(k, chunks available)` prediction chunks, so all 400 episodes remain in the AUC denominator. `chunk_k` uses only episodes that actually reached that individual zero-based chunk and prints that denominator.

In [ ]:
MAX_PREFIX_CHUNKS = 6
chunk_scores = (steps.groupby(['rollout_id', 'chunk_idx'], as_index=False).u_mean.mean()
                .sort_values(['rollout_id', 'chunk_idx']))
baseline_outcomes = observed[['rollout_id', 'suite', 'success', 'u_mean_episode']].copy()
baseline_outcomes['fail'] = (~baseline_outcomes.success.astype(bool)).astype(int)

score_catalog = {'whole_episode': baseline_outcomes[['rollout_id', 'u_mean_episode']].rename(
    columns={'u_mean_episode': 'score'})}
prefix_auc_rows = []
for k in range(1, MAX_PREFIX_CHUNKS + 1):
    score = (chunk_scores[chunk_scores.chunk_idx.lt(k)]
             .groupby('rollout_id', as_index=False).u_mean.mean()
             .rename(columns={'u_mean': 'score'}))
    assert score.rollout_id.nunique() == 400, f'prefix {k} lost baseline episodes'
    score_catalog[f'prefix_{k}_chunks'] = score
    joined = baseline_outcomes.merge(score, on='rollout_id', validate='one_to_one')
    prefix_auc_rows.append({'score_name': f'prefix_{k}_chunks',
                            **bootstrap_auc(joined.fail, joined.score, n_boot=1000)})
prefix_auc_rows.append({'score_name': 'whole_episode',
                        **bootstrap_auc(baseline_outcomes.fail,
                                        baseline_outcomes.u_mean_episode, n_boot=1000)})
prefix_auc = pd.DataFrame(prefix_auc_rows)

individual_auc_rows = []
for chunk_idx, scores in chunk_scores[chunk_scores.chunk_idx.lt(MAX_PREFIX_CHUNKS)].groupby('chunk_idx'):
    joined = baseline_outcomes.merge(
        scores[['rollout_id', 'u_mean']].rename(columns={'u_mean': 'score'}),
        on='rollout_id', validate='one_to_one')
    individual_auc_rows.append({'chunk_idx_zero_based': int(chunk_idx),
                                'episodes_reaching_chunk': len(joined),
                                **bootstrap_auc(joined.fail, joined.score, n_boot=1000)})
individual_chunk_auc = pd.DataFrame(individual_auc_rows)

print('Cumulative first-k uncertainty as a failure detector (all 400 episodes)')
display(prefix_auc[['score_name', 'n', 'failures', 'roc_auc', 'roc_ci_low', 'roc_ci_high', 'pr_auc']].round(4))
print('Individual chunk uncertainty; denominator falls when episodes finish before that chunk')
display(individual_chunk_auc[[
    'chunk_idx_zero_based', 'episodes_reaching_chunk', 'failures',
    'roc_auc', 'roc_ci_low', 'roc_ci_high', 'pr_auc']].round(4))
prefix_auc.to_csv(OUTPUT / 'prefix_failure_auc.csv', index=False)
individual_chunk_auc.to_csv(OUTPUT / 'individual_chunk_failure_auc.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(prefix_auc.score_name, prefix_auc.roc_auc, marker='o')
axes[0].fill_between(np.arange(len(prefix_auc)), prefix_auc.roc_ci_low, prefix_auc.roc_ci_high, alpha=.2)
axes[0].axhline(.5, color='black', linestyle='--', linewidth=1)
axes[0].set(ylabel='Failure ROC-AUC', title='Cumulative first-k uncertainty', ylim=(0, 1))
axes[0].tick_params(axis='x', rotation=35)
axes[1].plot(individual_chunk_auc.chunk_idx_zero_based, individual_chunk_auc.roc_auc, marker='o')
axes[1].fill_between(individual_chunk_auc.chunk_idx_zero_based,
                     individual_chunk_auc.roc_ci_low, individual_chunk_auc.roc_ci_high, alpha=.2)
axes[1].axhline(.5, color='black', linestyle='--', linewidth=1)
axes[1].set(xlabel='Chunk index (zero-based)', ylabel='Failure ROC-AUC',
            title='Individual-chunk uncertainty', ylim=(0, 1))
fig.tight_layout()
fig.savefig(OUTPUT / 'chunk_failure_auc.png', dpi=180, bbox_inches='tight')
plt.show()

## 5. Exploratory uncertainty-window sweeps

For each window, episodes inside the window take the exactly matched intervention outcome; all others retain their observed/no-op outcome. `window_sr_all_episodes` and `delta_pp` therefore always use every identity in the named cohort. Windows are selected and evaluated on the same data, so top windows are exploratory.

In [ ]:
LOWER_GRID = np.linspace(0.0, 0.06, 25)
UPPER_GRID = np.linspace(0.01, 0.08, 25)

score_frames = {}
identity_lookup = observed[PAIR_KEYS + ['rollout_id']]
for score_name, score in score_catalog.items():
    merged = identity_lookup.merge(score, on='rollout_id', validate='one_to_one')
    assert len(merged) == 400, f'{score_name} does not cover all 400 identities'
    score_frames[score_name] = merged[PAIR_KEYS + ['score']]

sweep_rows = []
for comparison in overall_deltas.itertuples(index=False):
    condition = validated[validated.config_hash.eq(comparison.config_hash)]
    if comparison.cohort == 'all_identity':
        base_scope, condition_scope, min_refined = observed, condition, 20
    else:
        base_scope = observed[observed.full_ablation_member]
        condition_scope = condition[condition.full_ablation_member]
        min_refined = 10
    outcomes = pair_one_to_one(base_scope, condition_scope)
    baseline_sr = float(outcomes.success_baseline.astype(bool).mean())
    for score_name, score_frame in score_frames.items():
        paired = outcomes.merge(score_frame, on=PAIR_KEYS, validate='one_to_one')
        assert len(paired) == len(outcomes)
        b = paired.success_baseline.astype(bool).to_numpy()
        r = paired.success_condition.astype(bool).to_numpy()
        score = paired.score.to_numpy(float)
        for lower in LOWER_GRID:
            for upper in UPPER_GRID:
                if upper <= lower:
                    continue
                selected = np.isfinite(score) & (score >= lower) & (score <= upper)
                policy = np.where(selected, r, b)
                eligible = int(selected.sum()) >= min_refined
                sweep_rows.append({
                    'cohort': comparison.cohort,
                    'condition_label': comparison.condition_label,
                    'score_name': score_name,
                    'episodes_in_sr_denominator': len(policy),
                    'lower': float(lower), 'upper': float(upper),
                    'n_refined': int(selected.sum()), 'eligible': eligible,
                    'baseline_sr': baseline_sr,
                    'window_sr_all_episodes': float(policy.mean()) if eligible else np.nan,
                    'delta_pp': 100 * (float(policy.mean()) - baseline_sr) if eligible else np.nan,
                    'selected_F_to_S': int((selected & ~b & r).sum()),
                    'selected_S_to_F': int((selected & b & ~r).sum()),
                })
window_sweep = pd.DataFrame(sweep_rows)
eligible = window_sweep[window_sweep.eligible & window_sweep.delta_pp.notna()].copy()
top_windows = (eligible.sort_values(
    ['cohort', 'condition_label', 'score_name', 'delta_pp', 'n_refined', 'lower', 'upper'],
    ascending=[True, True, True, False, False, True, True])
    .groupby(['cohort', 'condition_label', 'score_name'], sort=False).head(5).copy())
top_windows['rank'] = top_windows.groupby(
    ['cohort', 'condition_label', 'score_name']).cumcount() + 1

print('Top five exploratory windows per condition and uncertainty horizon')
display(top_windows[[
    'cohort', 'condition_label', 'score_name', 'rank',
    'episodes_in_sr_denominator', 'lower', 'upper', 'n_refined',
    'window_sr_all_episodes', 'delta_pp', 'selected_F_to_S', 'selected_S_to_F'
]].round(4))
window_sweep.to_csv(OUTPUT / 'uncertainty_window_sweep.csv', index=False)
top_windows.to_csv(OUTPUT / 'top_uncertainty_windows.csv', index=False)

In [ ]:
broad_whole = window_sweep[(window_sweep.cohort.eq('all_identity'))
                           & (window_sweep.score_name.eq('whole_episode'))]
for label, group in broad_whole.groupby('condition_label', sort=True):
    matrix = group.pivot(index='lower', columns='upper', values='delta_pp').sort_index()
    fig, ax = plt.subplots(figsize=(7, 5))
    image = ax.imshow(matrix.to_numpy(), origin='lower', aspect='auto', cmap='RdYlGn',
                      extent=[matrix.columns.min(), matrix.columns.max(),
                              matrix.index.min(), matrix.index.max()])
    fig.colorbar(image, ax=ax, label='Whole-cohort SR change (percentage points)')
    ax.set(xlabel='Upper uncertainty bound', ylabel='Lower uncertainty bound',
           title=f'Whole-episode window sweep\n{label}')
    fig.tight_layout()
    safe = ''.join(ch if ch.isalnum() else '_' for ch in label).strip('_').lower()
    fig.savefig(OUTPUT / f'window_sweep_{safe}.png', dpi=180, bbox_inches='tight')
    plt.show()

best = top_windows[(top_windows.cohort.eq('all_identity')) & top_windows['rank'].eq(1)].copy()
fig, ax = plt.subplots(figsize=(10, 4.5))
score_order = [f'prefix_{k}_chunks' for k in range(1, MAX_PREFIX_CHUNKS + 1)] + ['whole_episode']
x = np.arange(len(score_order))
labels = best.condition_label.unique().tolist()
width = .8 / max(1, len(labels))
for i, label in enumerate(labels):
    group = best[best.condition_label.eq(label)].set_index('score_name').reindex(score_order)
    ax.bar(x - .4 + width / 2 + i * width, group.delta_pp, width, label=label)
ax.axhline(0, color='black', linewidth=1)
ax.set(xticks=x, xticklabels=score_order, ylabel='Best exploratory SR change (percentage points)',
       title='Best full-cohort uncertainty window by chunk horizon')
ax.tick_params(axis='x', rotation=30)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT / 'best_window_delta_by_chunk_horizon.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved tables and figures to:', OUTPUT.resolve())